# Импорты

In [1]:
!pip install catboost
!pip install pymorphy3
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 8.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import hstack
import optuna

import pymorphy3
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

In [3]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Загрузка и исследование данных

In [4]:
df = pd.read_excel('nerd_analytics_v25.xlsx', sheet_name='reviews')

In [5]:
df = df[['id', 'comment', 'sentiment', 'rating', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']]

In [6]:
df.shape[0]

1235

In [7]:
df[['id', 'comment', 'sentiment', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']].describe()

,id,comment,sentiment,keywords_positive,keywords_neutral,keywords_negative,product,final_category
count,1235,1235,1235,815,397,189,1235,1235
unique,1235,65,3,29,12,16,6,5
top,266a542e-a9f3-4127-916e-76e017b10e1b,"Ответили быстро, не пришлось долго ждать",positive,приятно,целом,долго,аналитический модуль,качество ответа
freq,1,45,770,59,66,69,249,286


In [8]:
df.describe()

,rating
count,1235.000000
mean,3.710121
std,1.155714
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1235 entries, 0 to 1234
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1235 non-null   object
 1   comment            1235 non-null   object
 2   sentiment          1235 non-null   object
 3   rating             1235 non-null   int64 
 4   keywords_positive  815 non-null    object
 5   keywords_neutral   397 non-null    object
 6   keywords_negative  189 non-null    object
 7   product            1235 non-null   object
 8   final_category     1235 non-null   object
dtypes: int64(1), object(8)
memory usage: 87.0+ KB


In [10]:
df['product'].value_counts()

,count
product,
аналитический модуль,249
платёжный сервис,210
API интеграция,206
мобильное приложение,204
веб-сервис,193
личный кабинет,173


In [11]:
df.isna().sum(), df.duplicated().sum()

(id                      0
 comment                 0
 sentiment               0
 rating                  0
 keywords_positive     420
 keywords_neutral      838
 keywords_negative    1046
 product                 0
 final_category          0
 dtype: int64,
 np.int64(0))

# Определение общей тональности отзыва

## Подготовка данных

In [12]:
df['sentiment'].value_counts()

,count
sentiment,
positive,770
neutral,269
negative,196


In [13]:
df1 = df.copy()
RANDOM_STATE = 67

In [14]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = stopwords.words('russian')
bad_stopwords = ['не', 'нет', 'ни', 'без']

russian_stopwords = [
    word for word in russian_stopwords
    if word not in bad_stopwords
]

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)

df1['clean_comment'] = df1['comment'].apply(clean_text)

## Перебор моделей и гиперпараметров

In [16]:
sent_df = df1[['clean_comment', 'sentiment']].copy()
X_all = sent_df['clean_comment']
y_all = sent_df['sentiment']
groups_all = sent_df['clean_comment']

N_SPLITS = 5
CV_RANDOM_STATES = [RANDOM_STATE, 101, 202, 303, 404]


def build_features(train_part, valid_part):
    word_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, max_df=0.95)
    char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=20000)

    X_train_word = word_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_word = word_vectorizer.transform(valid_part['clean_comment'])
    X_train_char = char_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_char = char_vectorizer.transform(valid_part['clean_comment'])

    return hstack([X_train_word, X_train_char]), hstack([X_valid_word, X_valid_char])


def build_model_and_params(trial):
    model_type = trial.suggest_categorical('model_type', ['SVC', 'LogReg', 'CatBoost'])

    if model_type == 'SVC':
        params = {
            'C': trial.suggest_float('svc_C', 0.1, 5.0, log=True),
            'kernel': trial.suggest_categorical('svc_kernel', ['linear', 'rbf']),
            'gamma': 'scale',
            'class_weight': trial.suggest_categorical('svc_class_weight', [None, 'balanced']),
        }
        return model_type, params

    if model_type == 'LogReg':
        params = {
            'C': trial.suggest_float('lr_C', 0.1, 10.0, log=True),
            'solver': trial.suggest_categorical('lr_solver', ['lbfgs', 'saga']),
            'class_weight': trial.suggest_categorical('lr_class_weight', [None, 'balanced']),
            'max_iter': 3000,
            'n_jobs': -1,
            'random_state': RANDOM_STATE,
        }
        return model_type, params

    params = {
        'iterations': trial.suggest_int('cb_iterations', 200, 900),
        'depth': trial.suggest_int('cb_depth', 4, 10),
        'learning_rate': trial.suggest_float('cb_learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_int('cb_l2_leaf_reg', 1, 12),
        'random_strength': trial.suggest_float('cb_random_strength', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('cb_bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('cb_border_count', 64, 255),
        'auto_class_weights': trial.suggest_categorical('cb_auto_class_weights', ['Balanced', None]),
    }
    return model_type, params


def make_model(model_type, model_params):
    if model_type == 'SVC':
        return SVC(**model_params)

    if model_type == 'LogReg':
        return LogisticRegression(**model_params)

    return CatBoostClassifier(
        **model_params,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        random_state=RANDOM_STATE,
        task_type='GPU',
        verbose=False,
    )


def get_cv_scores(model_type, model_params, data, cv_random_states):
    fold_scores = []

    for cv_seed in cv_random_states:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

        for tr_fold_idx, val_fold_idx in cv.split(
            data['clean_comment'],
            data['sentiment'],
            groups=data['clean_comment'],
        ):
            tr_part = data.iloc[tr_fold_idx]
            val_part = data.iloc[val_fold_idx]

            X_tr, X_val = build_features(tr_part, val_part)
            y_tr = tr_part['sentiment']
            y_val = val_part['sentiment']

            model = make_model(model_type, model_params)
            model.fit(X_tr, y_tr, verbose=False) if model_type == 'CatBoost' else model.fit(X_tr, y_tr)
            pred = model.predict(X_val).flatten() if model_type == 'CatBoost' else model.predict(X_val)

            fold_scores.append(f1_score(y_val, pred, average='macro'))

    return fold_scores


def objective(trial):
    model_type, model_params = build_model_and_params(trial)
    fold_scores = get_cv_scores(model_type, model_params, sent_df, CV_RANDOM_STATES)
    return float(np.mean(fold_scores))


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=40)

print('BEST MEAN CV SCORE:')
print(study.best_value)
print('BEST PARAMS:')
print(study.best_params)


[I 2026-05-25 15:40:32,484] A new study created in memory with name: no-name-ae1e9a8f-165f-433f-be22-2c376aecc19c
[I 2026-05-25 15:40:46,737] Trial 0 finished with value: 0.3561827742976822 and parameters: {'model_type': 'LogReg', 'lr_C': 0.4604512348683669, 'lr_solver': 'saga', 'lr_class_weight': 'balanced'}. Best is trial 0 with value: 0.3561827742976822.
[I 2026-05-25 15:42:26,913] Trial 1 finished with value: 0.3573502767178464 and parameters: {'model_type': 'CatBoost', 'cb_iterations': 614, 'cb_depth': 5, 'cb_learning_rate': 0.025699473441646203, 'cb_l2_leaf_reg': 3, 'cb_random_strength': 4.152879163423533, 'cb_bagging_temperature': 2.5724281189423115, 'cb_border_count': 66, 'cb_auto_class_weights': None}. Best is trial 1 with value: 0.3573502767178464.
[I 2026-05-25 15:42:38,225] Trial 2 finished with value: 0.35229516373824277 and parameters: {'model_type': 'LogReg', 'lr_C': 1.1463487962934191, 'lr_solver': 'saga', 'lr_class_weight': 'balanced'}. Best is trial 1 with value: 0.35

BEST MEAN CV SCORE:
0.4230125221044272
BEST PARAMS:
{'model_type': 'CatBoost', 'cb_iterations': 728, 'cb_depth': 4, 'cb_learning_rate': 0.06802738881986581, 'cb_l2_leaf_reg': 4, 'cb_random_strength': 9.918716096056352, 'cb_bagging_temperature': 7.205067472832152, 'cb_border_count': 208, 'cb_auto_class_weights': 'Balanced'}


## Обучение лучшей модели

In [17]:
best_params = dict(study.best_params)
best_model_type = best_params.pop('model_type')

if best_model_type == 'SVC':
    model_params = {
        'C': best_params['svc_C'],
        'kernel': best_params['svc_kernel'],
        'gamma': 'scale',
        'class_weight': best_params['svc_class_weight'],
    }
elif best_model_type == 'LogReg':
    model_params = {
        'C': best_params['lr_C'],
        'solver': best_params['lr_solver'],
        'class_weight': best_params['lr_class_weight'],
        'max_iter': 3000,
        'n_jobs': -1,
        'random_state': RANDOM_STATE,
    }
else:
    model_params = {
        'iterations': best_params['cb_iterations'],
        'depth': best_params['cb_depth'],
        'learning_rate': best_params['cb_learning_rate'],
        'l2_leaf_reg': best_params['cb_l2_leaf_reg'],
        'random_strength': best_params['cb_random_strength'],
        'bagging_temperature': best_params['cb_bagging_temperature'],
        'border_count': best_params['cb_border_count'],
        'auto_class_weights': best_params['cb_auto_class_weights'],
    }

cv_results = []
y_true_all = []
y_pred_all = []

for cv_seed in CV_RANDOM_STATES:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

    for fold, (tr_fold_idx, val_fold_idx) in enumerate(
        cv.split(X_all, y_all, groups=groups_all),
        start=1,
    ):
        tr_part = sent_df.iloc[tr_fold_idx]
        val_part = sent_df.iloc[val_fold_idx]

        X_tr, X_val = build_features(tr_part, val_part)
        y_tr = tr_part['sentiment']
        y_val = val_part['sentiment']

        model = make_model(best_model_type, model_params)
        model.fit(X_tr, y_tr, verbose=False) if best_model_type == 'CatBoost' else model.fit(X_tr, y_tr)
        pred = model.predict(X_val).flatten() if best_model_type == 'CatBoost' else model.predict(X_val)

        cv_results.append({
            'seed': cv_seed,
            'fold': fold,
            'accuracy': accuracy_score(y_val, pred),
            'macro_f1': f1_score(y_val, pred, average='macro'),
        })
        y_true_all.extend(y_val)
        y_pred_all.extend(pred)

cv_results_df = pd.DataFrame(cv_results)
y_true_cv = pd.Series(y_true_all, name='true')
y_pred_cv = pd.Series(y_pred_all, name='pred')

X_full, _ = build_features(sent_df, sent_df)
best_model = make_model(best_model_type, model_params)
best_model.fit(X_full, y_all, verbose=False) if best_model_type == 'CatBoost' else best_model.fit(X_full, y_all)

CatBoostClassifier(auto_class_weights='Balanced', bagging_temperature=7.205067472832152, border_count=208, depth=4, eval_metric='TotalF1', iterations=728, l2_leaf_reg=4, learning_rate=0.06802738881986581, loss_function='MultiClass', random_state=67, random_strength=9.918716096056352, task_type='GPU', verbose=False)

In [18]:
accuracy_mean = cv_results_df['accuracy'].mean()
accuracy_std = cv_results_df['accuracy'].std()
macro_f1_mean = cv_results_df['macro_f1'].mean()
macro_f1_std = cv_results_df['macro_f1'].std()

print(f'\nBest model type: {best_model_type}')
print(f'CV seeds: {CV_RANDOM_STATES}')
print(f'Accuracy: {accuracy_mean:.4f} +/- {accuracy_std:.4f}')
print(f'Macro F1: {macro_f1_mean:.4f} +/- {macro_f1_std:.4f}')
print('\nFOLD METRICS SUMMARY:\n')
print(cv_results_df[['accuracy', 'macro_f1']].describe())
print('\nCLASSIFICATION REPORT OVER ALL CV PREDICTIONS:\n')
print(classification_report(y_true_cv, y_pred_cv))
print('\nCONFUSION MATRIX OVER ALL CV PREDICTIONS:\n')
print(confusion_matrix(y_true_cv, y_pred_cv))


Best model type: CatBoost
CV seeds: [67, 101, 202, 303, 404]
Accuracy: 0.5000 +/- 0.1120
Macro F1: 0.4230 +/- 0.1125

FOLD METRICS SUMMARY:

        accuracy   macro_f1
count  25.000000  25.000000
mean    0.500030   0.423013
std     0.111998   0.112493
min     0.277311   0.213857
25%     0.452055   0.348485
50%     0.519481   0.439942
75%     0.571429   0.487475
max     0.669604   0.666986

CLASSIFICATION REPORT OVER ALL CV PREDICTIONS:

              precision    recall  f1-score   support

    negative       0.25      0.45      0.33       980
     neutral       0.38      0.37      0.37      1345
    positive       0.69      0.56      0.62      3850

    accuracy                           0.50      6175
   macro avg       0.44      0.46      0.44      6175
weighted avg       0.55      0.50      0.52      6175


CONFUSION MATRIX OVER ALL CV PREDICTIONS:

[[ 442  120  418]
 [ 297  491  557]
 [1001  686 2163]]


## Сохранение результата

In [19]:
best_model.save_model('reviews_sentiments_comment_only.cbm')